In [ ]:
include( "../geom.jl" )
include( "../kernel.jl" )

include( "../0-params/rho-list.jl" )

using ThreadsX

_, base = expfolder( "system" )

runsim = false;

In [ ]:
# Color gradient for plotting.
colormap = cgrad( :ice )

In [ ]:
# Speed list to consider.
slist = [0.01,0.1,1.0]
Ns = length( slist )

# System sizes to consider.
N = 128;  N̂ = 128

# Extract critical noise.
folderdata = [findfolder( N, ρlist[j], slist[i]; base=base, crit=true ) for i ∈ 1:Ns, j ∈ 1:Nρ]
f̂olderdata = [findfolder( N̂, ρlist[j], slist[i]; base=base, crit=true ) for i ∈ 1:Ns, j ∈ 1:Nρ]

# Unpack the kernel parameters.
f0 = 100.0
γdata = vec.( readdlm.( folderdata.*"kernel-parameters-1.txt" ) )
c0data = first.( γdata )
println( "Kernel parameters: ", c0data )

In [ ]:
# Import area and perimeter scaling values.
βdata = first.( vec.( readdlm.( folderdata.*"area-perimeter-scaling.txt" ) ), 2 )

# Import the cluster change frequency values.
θdata = first.( readdlm.( folderdata.*"optimal-rates-hat.txt" )', 3 )'

# Extract the power law.
λdata = first.( readdlm.( folderdata.*"power-law.txt" ) );

In [ ]:
# Import cluster mass distribution extracted from data.
Fdata = map.( F -> round.( defInt, F ), vec.( readdlm.( f̂olderdata.*"cluster-frequency.txt" ) ) )
fdata = [Fdata[i,j]./sum( Fdata[i,j] ) for i ∈ 1:Ns, j ∈ 1:Nρ]

# Unpack the probability map data.
ℓdata = [polyloglaw.( 1:N̂; β=βdata[i,j] ) for i ∈ 1:Ns, j ∈ 1:Nρ]
Adata = [(ℓdata[i,j].^2)./(4π) for i ∈ 1:Ns, j ∈ 1:Nρ]

# Construct the probability map.
Ωdata = [ThreadsX.map(
    j -> [P( k2; N=N̂, i=k1, λℓ=θdata[i,j][1], λr=θdata[i,j][2], κ=θdata[i,j][3] ) for k1 in 1:N̂, k2 in 1:N̂], 1:Nρ ) for i ∈ 1:Ns];

In [ ]:
# Initialize progress meter.
meter = Progress( Ns*Nρ, "Simulating master equations."; dt=1.0 )

# Initialize data variables.
T = 10.0
f̂data = Matrix{Vector{defFloat}}( undef, Ns, Nρ )

# Simulate the master equations in the extrapolated system size.
for i ∈ 1:Ns, j ∈ 1:Nρ
    f̂data[i,j] = runsim ?
        kernelloss( Fdata[i,j], γdata[i,j], (N̂, Ωdata[i][j], Adata[i,j], ℓdata[i,j]); T=T, verbose=true )[2] :
        vec( readdlm( folderdata[i,j]*"cluster-distribution-hat.txt" ) )
    next!( meter )
end

# Compute the log-likelihood of the steady state distributions.
λ̂data = [10.0^optimize( createzipfloss( N̂, N̂.*f̂data[i,j] ), [log10( 1.5 )] ).minimizer[1]
    for i ∈ 1:Ns, j ∈ 1:Nρ];

In [ ]:
# Select density to illustrate.
i = 3;  j = 15
f = fdata[i,j]
f̂ = f̂data[i,j]

# Plot bounds.
xmax = ceil( log10( N̂ ) )
ymin = floor( minimum( log10.( f[f .> 0] ) ) )

# Plot the final distribution from the mean-field equations.
plt = plot(; size=(350,200), legend=false )

# Population size.
plot!( plt, [N̂, N̂], 10.0.^[ymin,0]; color=:black, linestyle=:dot, lw=2, label="system size" )

# Plot mean-field and simulation cluster mass distributions.
scatter!( plt, f; color=:gray, label="raw data" )
scatter!( plt, f̂; color=:black, markersize=3, lw=2, label="master eq." )

plot!( plt; xlims=10.0.^[0,xmax], xscale=:log10 )
plot!( plt; ylims=10.0.^[ymin,0], yscale=:log10 )

plot!( plt; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )

# saveplot( plt, "../../figures/extrapolated-cmd_N-$(N)"; dpi=600 )

In [ ]:
# Abundance of the measured distribution.
Hdata = sum.( map( f -> 0.0 .< f, fdata ) )./N

# Compute the power-law likeness metric across the fits.
σdata = jsd.( fdata, f̂data )
σmin, σmax = extrema( σdata )
ςdata = (σdata .- σmin)./(σmax - σmin)

In [ ]:
j = 16
markerlist = [:circ, :dtriangle, :square]

# Plot the difference in the scaling exponent between model and data.
plt = plot(; layout=(2,1), size=(400,450), legend=:outerright )
plot!( plt; right_margin=10pt, left_margin=0pt, bottom_margin=-5pt )

# Plot the cluster mass distribution for three examples in the small size case.
# plot!( plt[1], )

for i ∈ 1:Ns
    # Unpack the distributions to compare.
    f = fdata[i,j];  f̂ = f̂data[i,j]
    color = colormap[i/(Ns+1)]

    # Plot the distributions.
    scatter!( plt, abs.( f .- f̂ ); color=color, marker=markerlist[i], markerstrokewidth=0,
        label=latexstring( "s=10^{$(round( defInt, log10( slist[i] ) ))}" ) )
    # scatter!( plt, f̂; color=:black, lw=2, label="" )
end

xmax = log10( 128 );  xticks = round.( defInt, 2.0.^(0:1:7) )
ymin = -8;  yticks = 10.0.^(0:-2:ymin)

plot!( plt[1]; xlims=10.0.^[0,xmax], xscale=:log10, xticks=(xticks,string.( xticks )) )
plot!( plt[1]; ylims=10.0.^[ymin,0], yscale=:log10, yticks=yticks )
plot!( plt[1]; xlabel="cluster size, "*L"i", ylabel="probability difference" )
plot!( plt[1]; title="(a)", titleposition=:left )

# Plot the ratio-likeness relationship.
for i ∈ 1:Ns
    xlist = c0data[i,:]./f0
    ylist = Hdata[i,:].*(1.0 .- ςdata[i,:])

    scatter!( plt[2], xlist, ylist;
        color=colormap[i/(Ns+1)], lw=2, marker=markerlist[i],
        markerstrokewidth=0, label="" )

    # Extract linear relationship between the rate ratio and the likeness.
    μ = [ones( length( xlist ) ) xlist] \ ylist

    # Plot the linear relationship.
    xmin, xmax = 0, 0.1
    plot!( plt[2], [xmin, xmax], μ[1] .+ μ[2].*[xmin,xmax];
        color=colormap[i/(Ns+1)], lw=2, label="" )
end

plot!( plt[2]; xlims=(0,0.1+eps()), xscale=:linear )
plot!( plt[2]; ylims=(0,1) )
plot!( plt[2]; xlabel="coagulation-fragmentation ratio, "*L"c_0/f_0", ylabel="power-law likeness, "*L"ς" )
plot!( plt[2]; title="(b)", titleposition=:left )

# saveplot( plt, figurefolder*"coagulation-fragmentation-ratio_N-$(N).png"; dpi=600 )

In [ ]:
# Save the simulated distribution.
if false && runsim
    for i ∈ 1:Ns, j ∈ 1:Nρ
        writedlm( folderdata[i,j]*"cluster-distribution-hat.txt", f̂data[i,j] )
    end
end